# Multiple Linear Regression — Inflation Prediction (n → n+1)

**Predictors (period n):** BI Rate and M2_normalized  
**Target (period n+1):** Inflation YoY predicted one month ahead

## 1. Load & Align Data

In [ ]:
import json
import pandas as pd

month_map = {
    'Januari':1, 'Februari':2, 'Maret':3, 'April':4, 'Mei':5, 'Juni':6,
    'Juli':7, 'Agustus':8, 'September':9, 'Oktober':10, 'November':11, 'Desember':12
}

# Load raw data
inf = pd.DataFrame(json.load(open('./data/Inflation_YoY_2017_2026.json')))
m2  = pd.DataFrame(json.load(open('./data/M2M1_Norm_2017_2026.json')))
bi  = pd.DataFrame(json.load(open('./data/BI_Rate_2017_2026.json')))

# --- Inflation: period "Juli 2026" -> year-month ---
inf['month'] = inf['Periode'].str.extract(r'([A-Za-z]+)\s+')[0].map(month_map)
inf['year']  = inf['Periode'].str.extract(r'\S+\s+(\d+)')[0].astype(int)
inf['ym']    = inf['year'] * 100 + inf['month']

# --- M2: year/month -> year-month (use M2_normalized as the predictor) ---
m2['ym'] = m2['year'] * 100 + m2['month']
m2 = m2[['ym', 'M2_normalized']]

# --- BI Rate: decision date "19 Agustus 2026" -> year-month ---
bi['month'] = bi['Tanggal'].str.extract(r'\S+\s+([A-Za-z]+)\s+')[0].map(month_map)
bi['year']  = bi['Tanggal'].str.extract(r'\S+\s+\S+\s+(\d+)')[0].astype(int)
bi['ym']    = bi['year'] * 100 + bi['month']

# Effective BI Rate for a month = the rate set at the last decision of that month
bi_rate_monthly = bi.sort_values('ym').groupby('ym')['BI Rate'].last().rename('BI_Rate')

# Merge predictors (BI Rate and M2_normalized) at period n, onto a monthly grid
base = pd.DataFrame({'ym': sorted(set(inf['ym']) | set(m2['ym']))})
df = base.merge(m2, on='ym', how='left').merge(bi_rate_monthly, on='ym', how='left')

print('Merged monthly grid:', len(df), 'periods |', df['ym'].min(), '->', df['ym'].max())
print(df.head())

## 2. Build Lagged (n+1) Dataset & Fit

In [ ]:
def next_ym(y):
    m = y % 100; yr = y // 100
    m += 1
    if m == 13:
        m = 1; yr += 1
    return yr * 100 + m

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Target = inflation at period n+1, predicted from predictors at period n.
# For predictors at ym=n, the target is Inflasi at next_ym(n+1).
inf_map = inf.set_index('ym')['Inflasi']
df_p = df.dropna(subset=['BI_Rate', 'M2_normalized']).copy()
df_p['Inflasi_n1'] = df_p['ym'].map(lambda y: inf_map.get(next_ym(y)))
df_p = df_p.dropna(subset=['Inflasi_n1'])

X = df_p[['BI_Rate', 'M2_normalized']].values
y = df_p['Inflasi_n1'].values

model = LinearRegression().fit(X, y)
b0, b1, b2 = model.intercept_, *model.coef_
r2 = r2_score(y, model.predict(X))

print('Number of observations:', len(df_p))
print('Predictor period n  :', df_p['ym'].min(), '->', df_p['ym'].max())
print('Target period n+1   : next month after the predictor period')
print(f'Equation:  Inflasi_(n+1) = {b0:.4f} + ({b1:.4f}) * BI_Rate_n + ({b2:.4f}) * M2_normalized_n')
print(f'R^2 = {r2:.4f}')

## 3. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df_viz = df_p.copy()
pred = model.predict(X)

plt.rcParams['figure.dpi'] = 100

# 1) Correlation heatmap
plt.figure(figsize=(5, 4))
sns.heatmap(df_viz[['BI_Rate', 'M2_normalized', 'Inflasi_n1']].corr(), annot=True,
            cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap (n -> n+1)')
plt.tight_layout()
plt.show()

# 2) Actual vs Predicted
plt.figure(figsize=(6, 5))
plt.scatter(y, pred, alpha=0.6)
lims = [min(y.min(), pred.min()), max(y.max(), pred.max())]
plt.plot(lims, lims, 'r--')
plt.xlabel('Actual Inflation (n+1)')
plt.ylabel('Predicted Inflation')
plt.title(f'Actual vs Predicted (n+1)  ·  R^2 = {r2:.4f}')
plt.tight_layout()
plt.show()

In [ ]:
# 3) 3D scatter + fitted regression plane (2 predictors -> 1 target)
from mpl_toolkits.mplot3d import Axes3D

xx, yy = np.meshgrid(
    np.linspace(df_viz['BI_Rate'].min(), df_viz['BI_Rate'].max(), 20),
    np.linspace(df_viz['M2_normalized'].min(), df_viz['M2_normalized'].max(), 20)
)
zz = b0 + b1 * xx + b2 * yy

fig = plt.figure(figsize=(9, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df_viz['BI_Rate'], df_viz['M2_normalized'], y, c=y, cmap='viridis', s=25)
ax.plot_surface(xx, yy, zz, alpha=0.3, color='red')
ax.set_xlabel('BI Rate')
ax.set_ylabel('M2 normalized')
ax.set_zlabel('Inflation (n+1)')
ax.set_title('Fitted Plane: Inflasi_(n+1) = %.3f + %.3f*BI_rate + %.3f*M2' % (b0, b1, b2))
plt.tight_layout()
plt.show()

# 4) Residuals
resid = y - pred
plt.figure(figsize=(6, 4))
plt.scatter(pred, resid, alpha=0.6)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Inflation (n+1)')
plt.ylabel('Residual')
plt.title('Residuals (n -> n+1)')
plt.tight_layout()
plt.show()

## Summary — Model at period *n → n+1*

$$\text{Inflasi}_{n+1} = b_0 + b_1 \cdot \text{BI\_Rate}_n + b_2 \cdot \text{M2\_normalized}_n$$

The fitted coefficients and **R²** were printed by the training cell above.

- **BI Rate coefficient (n)** → how much next-month Inflation moves when this month's BI Rate rises by 1 point (holding M2 fixed)
- **M2_normalized coefficient (n)** → how much next-month Inflation moves when normalized M2 rises (holding BI Rate fixed)
- **R²** → share of next-month Inflation variation explained by this month's predictors